# Visual test: survival curve functions

Tests `_km_curve`, `_logrank_p`, `_bh_correct`, and `plot_survival_b` from `analysis_2f1p_multispec.py` using synthetic data with known properties.

**Expected behaviour by section:**
- **`_km_curve` shape tests** — step function starting at S=1; non-increasing; CI narrows early (many at risk) and widens late (few); all-censored stays flat at 1.0.
- **Two-group visual** — Fast group (early events) drops steeply then levels; Slow group drops later; censored tick marks (`+`) visible on the mixed condition.
- **`_logrank_p`** — clearly different groups → p ≪ 0.05; same-distribution groups → p ≫ 0.05.
- **`_bh_correct`** — adjusted values ≥ raw, monotone, in [0,1].
- **`plot_survival_b`** — full function renders; curves + CI ribbons + censored ticks + p-value text box all present.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'fish') if os.getcwd().endswith('notebooks') else os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from analysis_style import inline_display, set_style, COND_PALETTE, TIME_STEP_MS
from analysis_2f1p_multispec import (
    _km_curve,
    _logrank_p,
    _bh_correct,
    plot_survival_b,
    _Cfg,
)

print('Imports OK')

In [ ]:
# ── synthetic survival data ───────────────────────────────────────────────
rng = np.random.default_rng(7)

def uniform_events(n, lo, hi, rng):
    """All-event array: every episode gets an event time in [lo, hi] ms."""
    return rng.uniform(lo, hi, size=n), np.ones(n, dtype=int)

def mixed_events(n, lo, hi, p_event, max_time, rng):
    """Mixed: fraction p_event get events; rest are censored at max_time."""
    times  = np.full(n, float(max_time))
    events = np.zeros(n, dtype=int)
    mask = rng.random(n) < p_event
    times[mask]  = rng.uniform(lo, hi, size=mask.sum())
    events[mask] = 1
    return times, events

MAX_T = 500.0
N     = 40

t_fast,   e_fast   = uniform_events(N, 50,  150, rng)
t_slow,   e_slow   = uniform_events(N, 300, 450, rng)
t_same_a, e_same_a = uniform_events(N, 200, 300, rng)
t_same_b, e_same_b = uniform_events(N, 200, 300, rng)
t_mixed,  e_mixed  = mixed_events(N, 80, 200, 0.5, MAX_T, rng)
t_all_cen = np.full(N, MAX_T);  e_all_cen = np.zeros(N, dtype=int)

print(f'fast:    {e_fast.sum()}/{N} events, median t={np.median(t_fast[e_fast==1]):.0f} ms')
print(f'slow:    {e_slow.sum()}/{N} events, median t={np.median(t_slow[e_slow==1]):.0f} ms')
print(f'mixed:   {e_mixed.sum()}/{N} events (rest censored at {MAX_T:.0f} ms)')
print(f'all_cen: {e_all_cen.sum()}/{N} events (all censored)')

## 1. `_km_curve` — shape verification

Four conditions on shared axes. Check:
- All curves start at S=1, are non-increasing step functions
- Fast (blue) drops to 0 early; Slow (red) drops to 0 late
- Mixed (yellow) has a flat tail with `+` censored tick marks
- All-censored (gray) stays flat at 1.0
- CI ribbons are narrower early and wider/missing late

In [ ]:
import tempfile

set_style()
fig, ax = plt.subplots(figsize=(4.5, 3.0))

scenarios = [
    ('Fast (50–150 ms, all events)',  t_fast,    e_fast,    '#4477AA'),
    ('Slow (300–450 ms, all events)', t_slow,    e_slow,    '#EE6677'),
    ('Mixed (50 % censored)',         t_mixed,   e_mixed,   '#CCBB44'),
    ('All censored (flat at 1)',      t_all_cen, e_all_cen, '#888888'),
]

for label, times, events, color in scenarios:
    t_km, s_km, lo_km, hi_km = _km_curve(times, events)
    ax.plot(t_km, s_km, color=color, lw=1.8, label=label)
    ax.fill_between(t_km, lo_km, hi_km, color=color, alpha=0.18)
    cens_t = times[events == 0]
    if len(cens_t):
        ax.plot(cens_t, np.interp(cens_t, t_km, s_km),
                '+', color=color, ms=5, mew=1.0, alpha=0.7, zorder=3)

ax.set_xlabel('Time (ms)')
ax.set_ylabel('Survival S(t)')
ax.set_ylim(-0.05, 1.08)
ax.legend(fontsize=6, frameon=False, loc='upper right')
sns.despine(ax=ax)
plt.tight_layout(pad=0.4)

_tmpdir = tempfile.mkdtemp()
tmp_png = os.path.join(_tmpdir, 'km_shapes.png')
fig.savefig(tmp_png, dpi=150, bbox_inches='tight')
plt.close(fig)

from IPython.display import Image
Image(tmp_png)

## 2. `_logrank_p` — significance detection

- Fast vs Slow → expect p ≪ 0.001
- Same_A vs Same_B → expect large p (n.s.)

In [ ]:
p_diff = _logrank_p(t_fast, e_fast, t_slow, e_slow)
p_same = _logrank_p(t_same_a, e_same_a, t_same_b, e_same_b)
p_mix  = _logrank_p(t_fast, e_fast, t_mixed, e_mixed)

print(f'Fast vs Slow    : p = {p_diff:.2e}  (expect ≪ 0.001)')
print(f'Same_A vs Same_B: p = {p_same:.3f}  (expect n.s.)')
print(f'Fast vs Mixed   : p = {p_mix:.3f}')

assert p_diff < 0.001, f'Fast vs Slow should be highly significant, got p={p_diff}'
assert p_same > 0.05,  f'Same distributions should not be significant, got p={p_same}'
print('\nAssertions passed.')

## 3. `_bh_correct` — FDR correction

Adjusted values must be ≥ raw, in [0,1], and handle edge cases.

In [ ]:
raw = np.array([0.001, 0.01, 0.04, 0.5])
adj = _bh_correct(raw)
print('raw     :', raw)
print('adjusted:', adj)

assert np.all(adj >= raw - 1e-12), 'BH adjusted values should be >= raw'
assert np.all((adj >= 0) & (adj <= 1))
assert len(_bh_correct([])) == 0
assert _bh_correct(np.array([0.0]))[0] == 0.0
print('\n_bh_correct assertions passed.')

## 4. `plot_survival_b` — full function with synthetic DataFrames

Three conditions: fast eaters, slow eaters, mixed (50 % censored).  
Expected: well-separated KM curves; `syn_mixed` has a flat tail with `+` tick marks; p-value text box shows *** for fast vs slow.

In [ ]:
def make_step_df(event_steps, max_steps):
    """One B-role row per episode; eating_event=True when episode had an event."""
    rows = []
    for ep, t in enumerate(event_steps):
        ate = not np.isnan(t)
        rows.append(dict(env_id=0, episode_index=ep,
                         time_step=int(t) if ate else max_steps,
                         role='B', eating_event=ate))
    return pd.DataFrame(rows)

MAX_STEPS = int(MAX_T / TIME_STEP_MS)

def ms_to_steps(t_ms, event):
    steps = t_ms / TIME_STEP_MS
    steps = steps.copy(); steps[event == 0] = np.nan
    return steps

step_dfs = {
    'syn_fast':  make_step_df(ms_to_steps(t_fast,  e_fast),  MAX_STEPS),
    'syn_slow':  make_step_df(ms_to_steps(t_slow,  e_slow),  MAX_STEPS),
    'syn_mixed': make_step_df(ms_to_steps(t_mixed, e_mixed), MAX_STEPS),
}
cfg = _Cfg(
    main_conditions=['syn_fast', 'syn_slow'],
    control_a='syn_ctrl_a',
    control_b='syn_mixed',
    cond_labels={'syn_fast': 'Fast (50–150 ms)', 'syn_slow': 'Slow (300–450 ms)',
                 'syn_mixed': 'Mixed (50% censored)'},
    cond_colors={'syn_fast': COND_PALETTE[0], 'syn_slow': COND_PALETTE[1],
                 'syn_mixed': COND_PALETTE[3]},
)
print(f'TIME_STEP_MS = {TIME_STEP_MS}, MAX_STEPS = {MAX_STEPS}')

### 4a. All three conditions (main + control_b)

In [ ]:
with inline_display():
    plot_survival_b(step_dfs, out_dir='.', conditions=None, cfg=cfg)

### 4b. Main conditions only (fast + slow)

In [ ]:
with inline_display():
    plot_survival_b(step_dfs, out_dir='.', conditions=cfg.main_conditions, cfg=cfg)

## 5. Edge cases

- **Missing conditions** → silent return, no plot
- **Single condition** → one curve, no pairwise p-values

In [ ]:
# Missing conditions — should return silently (no output)
with inline_display():
    plot_survival_b({'ghost': None}, out_dir='.', conditions=['ghost'], cfg=cfg)
print('Missing conditions: returned silently (no figure above) ✓')

# Single condition
cfg_single = _Cfg(
    main_conditions=['syn_fast'], control_a='na', control_b='na',
    cond_labels={'syn_fast': 'Fast only'},
    cond_colors={'syn_fast': COND_PALETTE[0]},
)
with inline_display():
    plot_survival_b(step_dfs, out_dir='.', conditions=['syn_fast'], cfg=cfg_single)
print('Single condition: rendered ✓')